# Introducción a la atención en Transformers

Este cuaderno implementa una versión pequeña de **self-attention** usando NumPy y visualiza los pesos de atención con Matplotlib. Se usa la frase `esta frase es una prueba pequeña` para mostrar cómo cada token calcula cuánto atender a los demás.

> Nota: los pesos se inicializan aleatoriamente, así que los resultados pueden cambiar en cada ejecución.

## 1. Frase y vocabulario

Separamos la frase en palabras, asignamos un identificador a cada palabra y convertimos la frase en una lista de IDs.

In [ ]:
import numpy as np

frase = "esta frase es una prueba pequeña"
palabras = frase.split()

vocabulario = {
    word: i
    for i, word in enumerate(palabras)
}

token_ids = [vocabulario[token] for token in palabras]
vocab_size = len(vocabulario)

print("Palabras:", palabras)
print("Vocabulario:", vocabulario)
print("Token IDs:", token_ids)

## 2. Dimensiones y embeddings

Cada token se representa con un vector de dimensión 4. En este ejemplo, los embeddings son aleatorios y solo sirven para ilustrar las operaciones.

In [ ]:
dimension = 4
d_k = 2

E = np.random.randn(vocab_size, dimension)
X = E[token_ids]

print("E (matriz de embeddings):", E.shape)
print("X (secuencia de embeddings):", X.shape)

## 3. Proyecciones Query, Key y Value

Proyectamos cada embedding en tres representaciones: **Q** (queries), **K** (keys) y **V** (values).

In [ ]:
W_Q = np.random.randn(dimension, d_k)
W_K = np.random.randn(dimension, d_k)
W_V = np.random.randn(dimension, d_k)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

## 4. Puntajes y pesos de atención

Calculamos los productos entre queries y keys, escalamos por `sqrt(d_k)` y aplicamos softmax por fila. Cada fila de la matriz resultante representa la distribución de atención de un token sobre todos los tokens.

In [ ]:
puntajes = (Q @ K.T) / np.sqrt(d_k)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

pesos_atencion = softmax(puntajes)

print("Puntajes:", puntajes.shape)
print("Pesos de atención:", pesos_atencion.shape)
print(pesos_atencion)

## 5. Salida de atención

La salida es la suma ponderada de los vectores **V**, usando los pesos de atención calculados para cada token.

In [ ]:
Y = pesos_atencion @ V

print("Salida Y:", Y.shape)
print(Y)

## 6. Visualización de los pesos

El mapa de calor muestra cuánto atiende cada token (fila, **Query**) a cada token (columna, **Key**).

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.imshow(pesos_atencion, cmap="viridis")

plt.xticks(range(len(palabras)), palabras, rotation=45)
plt.yticks(range(len(palabras)), palabras)
plt.xlabel("Tokens a los que se presta atención (Keys)")
plt.ylabel("Tokens que prestan atención (Queries)")
plt.title("Mapa de calor de los pesos de atención")

for i in range(len(palabras)):
    for j in range(len(palabras)):
        plt.text(
            j, i, f"{pesos_atencion[i, j]:.2f}",
            ha="center", va="center", color="white"
        )

plt.colorbar(label="Peso de atención")
plt.tight_layout()
plt.show()